# Evaluation: Logistic Regression on the 200k Dataset (cmu + synthetic)

## 1. Introduction

This notebook covers steps 4 (**Evaluate**) and 5 (**Predict**) for the 200k model. It follows `05_modeling_logreg/03_evaluate` section by section, so the two models are scored in exactly the same way. In this notebook we:

1. Load the model saved by `02_train` (it is **not retrained**) and the test set, which is opened for the first time here.
2. Predict on the test set and score the model at two thresholds: the default 0.50 and the threshold chosen in `02_train`.
3. Report accuracy, precision, recall, F1 and specificity, the confusion matrix, the classification report and the ROC curve.
4. Compare training and test scores, and compare the model with the baselines.
5. Measure the uncertainty, the calibration, the per-1,000 view and the behaviour across sex and age groups.
6. Use the model on new, hypothetical respondents.

### Rules for the test set
- The test set is `diabetes_model_test`: **the same 14,039 real people as for the 70k model.** This is what makes the comparison fair. It contains **no synthetic rows**.
- The model, its features and its threshold were **fixed in `02_train`**. Nothing is changed after looking at the results here.
- If a result is disappointing, we report and discuss it. We do not tune the model on the test set.
- The training table is read **only to score the saved model on it** and to fit the baselines.

### What a believable result looks like
The cross-validation scores in `02_train` (ROC AUC about 0.81, computed on real validation people) tell us what to expect: test scores close to them. A test score clearly *above* them would be a warning sign of leakage.

**Inputs:** `logreg_200k.joblib`, `diabetes_model_test`, `diabetes_cmu_synth_train` | **Output:** a results table for the model comparison notebook

## 2. Import Libraries and Connect

> **Model refresh note:** The saved evaluation commentary and outputs refer to the earlier feature set. Rerun the matching `02_train` notebook and then this notebook to refresh them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn
from pathlib import Path

from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, brier_score_loss, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score, roc_curve)

from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

engine = create_engine("mysql+pymysql://root:@localhost:3306/diabetes_final_db")

# Settings (the same names and values as in 02_train)
TRAIN_TABLE = "diabetes_cmu_synth_train"
TEST_TABLE = "diabetes_model_test"
RESULTS_TABLE = "diabetes_eval_logreg_200k"
TARGET = "Diabetes_binary"
RANDOM_STATE = 42
REAL_PREVALENCE = 0.14
MODEL_PATH = Path("../../application/models/logreg_200k.joblib")

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print("Libraries imported and database connection successful.")

## 3. Load the Saved Model and the Data

Checks on the way in: the scikit-learn version, and that **no test row is identical to a training row, real or synthetic**.

In [ ]:
bundle = joblib.load(MODEL_PATH)
model = bundle["model"]
FEATURES = bundle["features"]
THRESHOLD = bundle["threshold"]

train_df = pd.read_sql(f"SELECT * FROM {TRAIN_TABLE}", con=engine)
test_df = pd.read_sql(f"SELECT * FROM {TEST_TABLE}", con=engine)
X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]
is_real = (train_df["is_synthetic"] == 0).to_numpy()

# Leakage check: rows that appear identically in both tables (all 22 columns)
cols = list(test_df.columns)
shared = test_df.merge(train_df[cols].drop_duplicates(), how="inner")

loaded = pd.DataFrame({
    "Item": ["Saved model file", "Features used by the model", "Decision threshold from 02_train",
             "scikit-learn version (saved / current)", "Test rows", "Test: no diabetes / diabetes",
             "Training rows (read only, for scoring)", "of which synthetic",
             "Test rows identical to a training row (real or synthetic)"],
    "Value": [MODEL_PATH.name, len(FEATURES), round(THRESHOLD, 3),
              f"{bundle['sklearn_version']} / {sklearn.__version__}",
              f"{len(test_df):,}", f"{(y_test == 0).sum():,} / {(y_test == 1).sum():,}",
              f"{len(train_df):,}", f"{int(train_df['is_synthetic'].sum()):,}", len(shared)],
})
display(loaded.style.hide(axis="index"))

print("Model and data loaded successfully. The model is NOT retrained in this notebook.")

## 4. Evaluate. Predict on the Test Set

In [ ]:
proba_test = model.predict_proba(X_test)[:, 1]
proba_train = model.predict_proba(X_train)[:, 1]

chosen_label = f"Chosen ({THRESHOLD:.2f})"
thresholds_used = {"Default (0.50)": 0.5, chosen_label: THRESHOLD}
pred_test = {name: (proba_test >= t).astype(int) for name, t in thresholds_used.items()}

preview = pd.DataFrame({
    "Actual": y_test.head(5).values,
    "Probability of diabetes": proba_test[:5].round(3),
    "Predicted at default 0.50": pred_test["Default (0.50)"][:5],
    f"Predicted at {chosen_label}": pred_test[chosen_label][:5],
})
display(preview.style.hide(axis="index"))

print("Predictions on the test set completed.")

## 5. Accuracy, Precision, Recall and F1

The metrics are the same as in the 70k notebook. **Recall** is our main metric. Precision and accuracy are measured on balanced test data, so they are more optimistic than they would be on a real population. Recall, specificity and ROC AUC are not affected by the balance.

In [ ]:
def evaluate(y, proba, threshold):
    """All metrics for one set of rows at one threshold."""
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return {
        "Accuracy": accuracy_score(y, pred),
        "Recall (sensitivity)": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "Precision": tp / (tp + fp),
        "F1": f1_score(y, pred),
        "ROC AUC": roc_auc_score(y, proba),
        "Brier score": brier_score_loss(y, proba),
        "Missed diabetes cases (false negatives)": fn,
        "Healthy people flagged (false positives)": fp,
    }

results = pd.DataFrame({name: evaluate(y_test, proba_test, t) for name, t in thresholds_used.items()})

show = results.round(3).astype(object)
for r in ["Missed diabetes cases (false negatives)", "Healthy people flagged (false positives)"]:
    show.loc[r] = results.loc[r].astype(int).map("{:,}".format)
display(show)

**Interpretation:** This saved interpretation is from the previous feature set. Rerun this notebook to refresh it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, t) in zip(axes, thresholds_used.items()):
    cm = confusion_matrix(y_test, pred_test[name])
    pct = cm / cm.sum(axis=1, keepdims=True) * 100
    labels = np.array([[f"{cm[i, j]:,}\n({pct[i, j]:.1f}% of row)" for j in range(2)] for i in range(2)])
    sns.heatmap(cm, annot=labels, fmt="", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Predicted: no diabetes", "Predicted: diabetes"],
                yticklabels=["Actual: no diabetes", "Actual: diabetes"])
    ax.set_title(f"Test set, threshold {name}")
plt.tight_layout()
plt.show()

**Interpretation:** This saved interpretation is from the previous feature set. Rerun this notebook to refresh it.

In [ ]:
for name in thresholds_used:
    report = pd.DataFrame(classification_report(
        y_test, pred_test[name], target_names=["No diabetes", "Diabetes"], output_dict=True)).T
    print(f"Classification report, threshold {name}:")
    display(report.round(3))

**Interpretation:** This saved interpretation is from the previous feature set. Rerun this notebook to refresh it.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba_test)
auc = roc_auc_score(y_test, proba_test)

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot(fpr, tpr, color="#4C78A8", label=f"Logistic regression, 200k (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Chance (AUC = 0.500)")
for (name, t), color in zip(thresholds_used.items(), ["#999999", "#E45756"]):
    m = results[name]
    ax.scatter(1 - m["Specificity"], m["Recall (sensitivity)"], s=70, color=color, zorder=3,
               label=f"Threshold {name}")
ax.set_xlabel("False alarm rate (1 - specificity)")
ax.set_ylabel("Recall (sensitivity)")
ax.set_title("ROC curve (test set)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

print(f"ROC AUC on the test set: {auc:.3f}")

**Interpretation:** This saved interpretation is from the previous feature set. Rerun this notebook to refresh it.

In [ ]:
def scores_5050(y, proba, t):
    """Same metrics, with precision/F1/accuracy on a 50/50 scale (for the unbalanced real training rows)."""
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    tpr, tnr = tp / (tp + fn), tn / (tn + fp)
    prec = tpr / (tpr + (1 - tnr))
    return {"Accuracy": (tpr + tnr) / 2, "Recall (sensitivity)": tpr, "Precision": prec,
            "F1": 2 * prec * tpr / (prec + tpr), "ROC AUC": roc_auc_score(y, proba)}

metrics_to_compare = ["Accuracy", "Recall (sensitivity)", "Precision", "F1", "ROC AUC"]

rows = {}
for name, t in thresholds_used.items():
    train_all = evaluate(y_train, proba_train, t)
    train_real = scores_5050(y_train[is_real], proba_train[is_real], t)
    test_res = evaluate(y_test, proba_test, t)
    for metric in metrics_to_compare:
        rows.setdefault(metric, {})[f"Train (all 200k), {name}"] = train_all[metric]
        rows[metric][f"Train (real rows), {name}"] = train_real[metric]
        rows[metric][f"Test, {name}"] = test_res[metric]

gap_df = pd.DataFrame(rows).T
display(gap_df.round(3))

gaps = {m: gap_df.loc[m, f"Train (real rows), {chosen_label}"] - gap_df.loc[m, f"Test, {chosen_label}"]
        for m in gap_df.index}
print(f"Largest gap between the training (real rows) and test score at the chosen threshold: "
      f"{max(gaps.values(), key=abs):+.4f}")

**Interpretation:** This saved interpretation is from the previous feature set. Rerun this notebook to refresh it.

In [ ]:
baselines = {
    "Baseline: always no diabetes": DummyClassifier(strategy="constant", constant=0),
    "Baseline: flag everybody": DummyClassifier(strategy="constant", constant=1),
    "Baseline: random guess": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
}

comparison = {}
for name, m in baselines.items():
    m.fit(X_train, y_train)
    pred, proba = m.predict(X_test), m.predict_proba(X_test)[:, 1]
    comparison[name] = {"Accuracy": accuracy_score(y_test, pred), "Recall": recall_score(y_test, pred),
                        "Precision": precision_score(y_test, pred, zero_division=0),
                        "F1": f1_score(y_test, pred, zero_division=0), "ROC AUC": roc_auc_score(y_test, proba)}

for name in thresholds_used:
    r = results[name]
    comparison[f"Logistic regression, threshold {name}"] = {
        "Accuracy": r["Accuracy"], "Recall": r["Recall (sensitivity)"], "Precision": r["Precision"],
        "F1": r["F1"], "ROC AUC": r["ROC AUC"]}

display(pd.DataFrame(comparison).T.round(3))

**Interpretation:** This saved interpretation is from the previous feature set. Rerun this notebook to refresh it.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
y_arr, p_arr = y_test.to_numpy(), proba_test
n = len(y_arr)

boot = []
for _ in range(1000):
    idx = rng.integers(0, n, n)
    yb, pb = y_arr[idx], p_arr[idx]
    tn, fp, fn, tp = confusion_matrix(yb, (pb >= THRESHOLD).astype(int)).ravel()
    boot.append({"Recall": tp / (tp + fn), "Specificity": tn / (tn + fp),
                 "Precision": tp / (tp + fp), "ROC AUC": roc_auc_score(yb, pb)})
boot = pd.DataFrame(boot)

point = results[chosen_label]
ci = pd.DataFrame({
    "Estimate": [point["Recall (sensitivity)"], point["Specificity"], point["Precision"], point["ROC AUC"]],
    "95% CI low": boot.quantile(0.025).values,
    "95% CI high": boot.quantile(0.975).values,
}, index=["Recall", "Specificity", "Precision", "ROC AUC"])
display(ci.round(3))

**Interpretation:** This saved interpretation is from the previous feature set. Rerun this notebook to refresh it.

In [ ]:
frac_pos, mean_pred = calibration_curve(y_test, proba_test, n_bins=10, strategy="quantile")

calib = pd.DataFrame({"Mean predicted probability": mean_pred,
                      "Share who actually have diabetes": frac_pos})
calib.index = [f"Group {i}" for i in range(1, len(calib) + 1)]
display(calib.round(3))

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(mean_pred, frac_pos, marker="o", color="#4C78A8", label="Logistic regression, 200k")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfectly calibrated")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Share who actually have diabetes")
ax.set_title("Calibration (test set)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Brier score: {brier_score_loss(y_test, proba_test):.3f}")

**Interpretation:** This saved interpretation is from the previous feature set. Rerun this notebook to refresh it.

In [ ]:
rows = []
for name, t in thresholds_used.items():
    m = results[name]
    tpr, false_alarm_rate = m["Recall (sensitivity)"], 1 - m["Specificity"]
    n_pos = 1000 * REAL_PREVALENCE
    n_neg = 1000 - n_pos
    found = n_pos * tpr
    flagged_healthy = n_neg * false_alarm_rate
    rows.append({
        "Threshold": name,
        "Have diabetes": n_pos,
        "Found": found,
        "Missed": n_pos - found,
        "Healthy people flagged": flagged_healthy,
        "Total flagged": found + flagged_healthy,
        "Share with diabetes among those flagged": found / (found + flagged_healthy),
    })
display(pd.DataFrame(rows).round(2).style.hide(axis="index"))

**Interpretation:** This saved interpretation is from the previous feature set. Rerun this notebook to refresh it.

In [ ]:
test_all = test_df.copy()
test_all["proba"] = proba_test
test_all["pred"] = (proba_test >= THRESHOLD).astype(int)
test_all["Age group"] = pd.cut(test_all["Age"], bins=[0, 5, 9, 13],
                               labels=["18-44 (codes 1-5)", "45-64 (codes 6-9)", "65+ (codes 10-13)"])
test_all["Sex group"] = test_all["Sex"].map({0: "Female", 1: "Male"})

rows = []
for col in ["Sex group", "Age group"]:
    for level, g in test_all.groupby(col, observed=True):
        tn, fp, fn, tp = confusion_matrix(g[TARGET], g["pred"], labels=[0, 1]).ravel()
        rows.append({
            "Group": f"{col.replace(' group', '')}: {level}",
            "People": len(g),
            "% with diabetes": g[TARGET].mean() * 100,
            "Recall": tp / (tp + fn) if (tp + fn) else np.nan,
            "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "ROC AUC": roc_auc_score(g[TARGET], g["proba"]) if g[TARGET].nunique() == 2 else np.nan,
        })
display(pd.DataFrame(rows).round(3).style.hide(axis="index"))

**Interpretation:** This saved interpretation is from the previous feature set. Rerun this notebook to refresh it.

In [ ]:
def adjust_probability(p, prevalence=REAL_PREVALENCE):
    """Convert a probability from the balanced (50/50) scale to the real prevalence."""
    odds = p / (1 - p) * (prevalence / (1 - prevalence))
    return odds / (1 + odds)

def predict_risk(answers):
    row = pd.DataFrame([answers])[FEATURES]
    p = float(model.predict_proba(row)[:, 1][0])
    return {
        "Model probability (balanced data)": p,
        f"Adjusted to {REAL_PREVALENCE:.0%} prevalence": adjust_probability(p),
        "Flagged for follow-up": p >= THRESHOLD,
    }

people = {
    "Person A: 32 years old, healthy": {
        "HighBP": 0, "HighChol": 0, "CholCheck": 1, "BMI": 23, "Smoker": 0, "Stroke": 0,
        "HeartDiseaseorAttack": 0, "PhysActivity": 1, "Fruits": 1, "Veggies": 1, "HvyAlcoholConsump": 0,
        "AnyHealthcare": 1, "NoDocbcCost": 0, "GenHlth": 1, "MentHlth": 0, "PhysHlth": 0, "DiffWalk": 0,
        "Sex": 0, "Age": 3, "Education": 6, "Income": 7},
    "Person B: 57 years old, some risk factors": {
        "HighBP": 1, "HighChol": 0, "CholCheck": 1, "BMI": 31, "Smoker": 1, "Stroke": 0,
        "HeartDiseaseorAttack": 0, "PhysActivity": 1, "Fruits": 0, "Veggies": 1, "HvyAlcoholConsump": 0,
        "AnyHealthcare": 1, "NoDocbcCost": 0, "GenHlth": 3, "MentHlth": 2, "PhysHlth": 3, "DiffWalk": 0,
        "Sex": 1, "Age": 8, "Education": 4, "Income": 5},
    "Person C: 72 years old, several risk factors": {
        "HighBP": 1, "HighChol": 1, "CholCheck": 1, "BMI": 38, "Smoker": 0, "Stroke": 0,
        "HeartDiseaseorAttack": 1, "PhysActivity": 0, "Fruits": 0, "Veggies": 0, "HvyAlcoholConsump": 0,
        "AnyHealthcare": 1, "NoDocbcCost": 1, "GenHlth": 4, "MentHlth": 5, "PhysHlth": 15, "DiffWalk": 1,
        "Sex": 1, "Age": 11, "Education": 3, "Income": 2},
}
display(pd.DataFrame({name: predict_risk(a) for name, a in people.items()}).T.round(3))

**Interpretation:** This saved interpretation is from the previous feature set. Rerun this notebook to refresh it.

In [ ]:
out = results.T.reset_index().rename(columns={"index": "Threshold"})
out.insert(0, "Model", "Logistic regression (200k: real + synthetic)")
out.to_sql(RESULTS_TABLE, con=engine, if_exists="replace", index=False)

with engine.connect() as conn:
    saved_rows = conn.execute(text(f"SELECT COUNT(*) FROM {RESULTS_TABLE}")).scalar()

print(f"Results saved: {saved_rows} rows in '{RESULTS_TABLE}'.")

## 17. Summary

This notebook evaluates the current 200k model using the same 14 features and held-out test rows as the 70k model. The 200k training set has 143,845 more rows than the 70k training set: 71,922 synthetic diabetes rows and 71,923 additional real no-diabetes rows. Rerun this notebook after `02_train` to refresh its tables and charts.

The matched real-only control results are saved in `data/07_cmu+synthetic/diabetes_synthetic_control_comparison.csv`. In the current run, the real-only class-weighted control and the synthetic model have nearly identical test ROC AUC (0.8003 vs. 0.7999) and recall at their selected thresholds (0.8488 vs. 0.8487), so this run shows no measurable gain from the synthetic rows.